In [2]:
# Classification pipeline with stratified 5-fold cross-validation (preserves label proportions)
from sklearn.model_selection import StratifiedKFold
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.metrics import roc_curve
from sklearn.metrics import (
    classification_report, confusion_matrix, accuracy_score, roc_auc_score, balanced_accuracy_score,
    precision_score, recall_score, f1_score
)
from sklearn.preprocessing import StandardScaler
import numpy as np
import os
import json
import pandas as pd



In [3]:
# Helper function
def find_optimal_tresh(y_true, y_proba):
    fpr, tpr, thresholds = roc_curve(y_true, y_proba)
    J_scores = tpr - fpr
    best_idx = np.argmax(J_scores)
    return thresholds[best_idx]

def is_acc_signal(col):
    lc = col.lower()
    valid = (('gyr' in lc  )) and (not lc.endswith('num'))
    not_meta = col not in ['subject_id', 'patient_num', 'label', 'age', 'sex']
    return valid and not_meta
def specificity_score(y_true, y_pred):
    # Binary specificity = TN / (TN + FP)
    cm = confusion_matrix(y_true, y_pred)
    if cm.shape == (2, 2):
        tn, fp, fn, tp = cm.ravel()
        return tn / (tn + fp) if (tn + fp) else 0.0
    # Multiclass: macro-average of per-class specificity
    spec_per_class = []
    for i in range(cm.shape[0]):
        tn = cm.sum() - (cm[i, :].sum() + cm[:, i].sum() - cm[i, i])
        fp = cm[:, i].sum() - cm[i, i]
        spec_per_class.append(tn / (tn + fp) if (tn + fp) else 0.0)
    return float(np.mean(spec_per_class))


In [6]:
report_df_biobert = pd.read_csv(r'biobert_questionnaire_features.csv')
print(f"Report dataframe shape: {report_df_biobert.shape}")
report_df_raw = pd.read_csv('questionnaire_data_ML_wide.csv')
print(f"Report ML dataframe shape: {report_df_raw.shape}")
df_quant = pd.read_csv('ClinicalData.csv')
print(f"Clinical data shape: {df_quant.shape}")
Signal_df_method1 = pd.read_csv('wide_df')
print(f"Signal_method1 data shape: {Signal_df_method1.shape}")
Signal_df_method2 = pd.read_csv('extracted_features_method1.csv')
print(f"Signal_method2 data shape: {Signal_df_method2.shape}")

Report dataframe shape: (469, 769)
Report ML dataframe shape: (469, 31)
Clinical data shape: (469, 7)
Signal_method1 data shape: (469, 2025)
Signal_method2 data shape: (469, 32)


In [7]:
# Combine wide_df and report_df based on patient_num

model_type = 'multilabel'

if model_type == 'multilabel':
    combined_df = pd.merge(
        df_quant[['subject_id', 'label']],
        report_df_raw,
        on='subject_id',
        how='inner',
        suffixes=('', '_report')
    )
    combined_df = pd.merge(
        combined_df,
        Signal_df_method1,
        on='subject_id',
        how='inner',
        suffixes=('', '_signal')
    )

if model_type == 'Signal':
    combined_df = pd.merge(
        df_quant[['subject_id', 'label']],
        Signal_df_method1,
        on='subject_id',
        how='inner',
        suffixes=('', '_report')
    )

elif model_type == 'questionaire':
    combined_df = pd.merge(
        df_quant[['subject_id', 'label']],
        report_df_raw,
        on='subject_id',
        how='inner',
        suffixes=('', '_questionaire')
    )

print(f"Combined dataframe shape: {combined_df.shape}")
display(combined_df.head())



Combined dataframe shape: (469, 2056)


,subject_id,label,q_01,q_02,q_03,q_04,q_05,q_06,q_07,q_08,...,RelaxedTask_Left_AccVM_bp_3_7,RelaxedTask_Left_AccVM_bp_6_12,RelaxedTask_Left_AccVM_sentropy,RelaxedTask_Left_GyrVM_mean,RelaxedTask_Left_GyrVM_std,RelaxedTask_Left_GyrVM_rms,RelaxedTask_Left_GyrVM_domfreq,RelaxedTask_Left_GyrVM_bp_3_7,RelaxedTask_Left_GyrVM_bp_6_12,RelaxedTask_Left_GyrVM_sentropy
0,1,0,0,0,0,0,0,0,0,0,...,4.265345e-06,4.365871e-06,4.220537,0.043970,0.037095,0.057522,0.390625,0.000252,0.000193,3.247321
1,2,2,1,1,0,0,1,0,0,0,...,1.391696e-04,6.351266e-04,3.245044,0.209675,0.283024,0.352174,0.390625,0.005530,0.017788,3.179946
2,3,0,0,0,0,0,0,0,0,0,...,2.846572e-06,4.868614e-06,3.937279,0.020827,0.012405,0.024240,1.171875,0.000041,0.000045,3.583532
3,4,1,0,1,0,1,0,0,0,1,...,7.564303e-07,7.336435e-07,4.414410,0.015455,0.007694,0.017263,1.171875,0.000017,0.000014,3.824369
4,5,1,1,1,1,0,0,0,0,1,...,1.346834e-05,4.490356e-04,3.060737,0.102982,0.057209,0.117799,10.156250,0.000140,0.001711,3.302868


In [ ]:
import xgboost as xgb
from sklearn import svm

try:
    from xgboost import XGBClassifier
    xgb_available = True
except ImportError:
    print("XGBoost is not installed. Install with `pip install xgboost` to run XGBoost analysis.")
    xgb_available = False


clf_model = 'XGBoost'
temp_df = combined_df.copy()

if model_type != 'multilabel':
    temp_df = temp_df[temp_df['label'] != 0]  #0: HC, 1= PD, 2: Other disease
    temp_df['label'] = temp_df['label'].apply(lambda x : 0 if x == 2 else x)


x_train = temp_df.drop(columns=['label', 'subject_id'])
y_train = temp_df['label']  


scale_pos_weight = (y_train == 0).sum() / (y_train == 1).sum()

print(f"Filtered dataframe shape: {temp_df.shape}")
print(f"Label distribution:\n{temp_df['label'].value_counts().sort_index()}")


if clf_model == 'XGBoost':
    clf = xgb.XGBClassifier(
        objective="binary:logistic",
        eval_metric="auc",
        random_state=42,
        n_estimators=1000,
        learning_rate=0.01,
        max_depth=2,
        min_child_weight=2,
        subsample=0.8,
        colsample_bytree=0.8,
        gamma=0.1,
        reg_alpha=0.0,
        reg_lambda=10.0,
        scale_pos_weight=scale_pos_weight,
    )
elif clf_model == 'SVM':
    clf = svm.SVC(probability=True, random_state=42)


skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
xgb_acc_scores = []
xgb_roc_auc_scores = []
xgb_balanced_acc_scores = []
xgb_precision_scores = []
xgb_recall_scores = []
xgb_specificity_scores = []
xgb_f1_scores = []

for fold, (train_index, test_index) in enumerate(skf.split(x_train, y_train), 1):
    X_tr, X_te = x_train.iloc[train_index], x_train.iloc[test_index]
    y_tr, y_te = y_train.iloc[train_index], y_train.iloc[test_index]
    
    # scaler = StandardScaler()
    # X_tr_scaled = scaler.fit_transform(X_tr)
    # X_te_scaled = scaler.transform(X_te)


    clf.fit(X_tr, y_tr)
    
    if len(np.unique(y_train)) == 2:
        # Binary roc_auc
        y_proba = clf.predict_proba(X_te)[:,1]
        best_thresh = find_optimal_tresh (y_te, y_proba)
        roc_auc = roc_auc_score(y_te, y_proba)
        y_pred = (y_proba >= best_thresh).astype(int)
        precision = precision_score(y_te, y_pred)
        recall = recall_score(y_te, y_pred)
        specificity = specificity_score(y_te, y_pred)
        f1 = f1_score(y_te, y_pred)

    else:
        # Multiclass roc_auc
        y_proba = clf.predict_proba(X_te)
        # best_thresh = find_optimal_tresh (y_te, y_proba)
        roc_auc = roc_auc_score(y_te, y_proba, multi_class='ovr')
        y_pred = np.argmax(y_proba, axis=1)
        precision = precision_score(y_te, y_pred, average='weighted')
        recall = recall_score(y_te, y_pred, average='weighted')
        specificity = specificity_score(y_te, y_pred)
        f1 = f1_score(y_te, y_pred, average='weighted')
    
        
    acc = accuracy_score(y_te, y_pred)
    balanced_acc = balanced_accuracy_score(y_te, y_pred)


    xgb_acc_scores.append(acc)
    xgb_roc_auc_scores.append(roc_auc)
    xgb_balanced_acc_scores.append(balanced_acc)
    xgb_precision_scores.append(precision)
    xgb_recall_scores.append(recall)
    xgb_specificity_scores.append(specificity)
    xgb_f1_scores.append(f1)

    print(f"Fold {fold}: accuracy={acc:.4f}, roc_auc={roc_auc:.4f}, balanced_acc={balanced_acc:.4f}, precision={precision:.4f}, recall={recall:.4f}, specificity={specificity:.4f}, f1={f1:.4f}")
    print(classification_report(y_te, y_pred))
    print(confusion_matrix(y_te, y_pred))
    print('-' * 50)

print(f"Mean accuracy: {np.mean(xgb_acc_scores):.3f} (+/- {np.std(xgb_acc_scores):.4f})")
print(f"Mean ROC AUC: {np.mean(xgb_roc_auc_scores):.3f} (+/- {np.std(xgb_roc_auc_scores):.4f})")
print(f"Mean balanced accuracy: {np.mean(xgb_balanced_acc_scores):.3f} (+/- {np.std(xgb_balanced_acc_scores):.4f})")
print(f"Mean precision: {np.mean(xgb_precision_scores):.3f} (+/- {np.std(xgb_precision_scores):.4f})")
print(f"Mean recall: {np.mean(xgb_recall_scores):.3f} (+/- {np.std(xgb_recall_scores):.4f})")
print(f"Mean specificity: {np.mean(xgb_specificity_scores):.3f} (+/- {np.std(xgb_specificity_scores):.4f})")
print(f"Mean F1 score: {np.mean(xgb_f1_scores):.3f} (+/- {np.std(xgb_f1_scores):.4f})")

